# Kubernetes vs. Simulator Evaluation

This notebook reproduces the final evaluation cycle combining the Kubernetes replay
and the simulator runs. It loads the exported scheduling traces, harmonises the
schema, computes summary metrics, and visualises the trade-offs across policies with
Pareto-front analysis.

The workflow follows the architecture notes in Chapter 5.3.2 of the design document:

1. Select the runs and fetch the traces.
2. Harmonise the per-job schema and enrich it with timing features.
3. Optionally pull contextual metrics (left as hooks for Prometheus integration).
4. Produce visual comparisons and Pareto-front inspection.


## 1. Imports and helper utilities

The helpers below keep the notebook self-contained: they resolve the repository
root automatically, standardise datetime conversions, and expose a Pareto-front
filter for the multi-objective analysis.


In [1]:
from __future__ import annotations

from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")


def find_repo_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    for parent in [start, *start.parents]:
        if (parent / ".git").exists():
            return parent
    raise RuntimeError("Could not locate repository root from" f" {start}")


def safe_to_datetime(series: pd.Series) -> pd.Series:
    # Convert string timestamps to timezone-aware pandas datetimes. Placeholder
    # values such as 0001-01-01 are mapped to NaT so downstream metrics can drop them.
    cleaned = series.replace({"": pd.NA, "0001-01-01T00:00:00Z": pd.NA})
    return pd.to_datetime(cleaned, utc=True, errors="coerce")


def compute_wait_seconds(submit: pd.Series, start: pd.Series) -> pd.Series:
    wait = (start - submit).dt.total_seconds()
    return wait


def compute_runtime_seconds(start: pd.Series, end: pd.Series) -> pd.Series:
    runtime = (end - start).dt.total_seconds()
    return runtime


def pareto_front(points: np.ndarray) -> np.ndarray:
    # Return a boolean mask for the Pareto-efficient points (minimisation).
    if points.ndim != 2:
        raise ValueError("Pareto front expects a 2D array")
    n_points = points.shape[0]
    is_efficient = np.ones(n_points, dtype=bool)
    for i in range(n_points):
        if not is_efficient[i]:
            continue
        dominated = np.all(points >= points[i], axis=1) & np.any(points > points[i], axis=1)
        is_efficient &= ~dominated
    return is_efficient


repo_root = find_repo_root()
figures_dir = repo_root / "analysis" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

repo_root, figures_dir


(PosixPath('/home/goncalo/KubeEnergyScheduler'),
 PosixPath('/home/goncalo/KubeEnergyScheduler/analysis/figures'))

## 2. Locate experiment artefacts

Both the simulator and the Kubernetes replay export CSV/JSON artefacts. Update the
paths below if you want to compare different runs.


In [2]:
# Simulator artefacts (per-job traces  summary metrics)
sim_dir = repo_root / "kubenergysched" / "results_latest"
sim_summary_path = sim_dir / "summary.csv"

# Kubernetes replay decisions (JSONL  optional CSV projection)
k8_decisions_jsonl = repo_root / "decisions.jsonl"
k8_decisions_csv = repo_root / "decisions.csv"

sim_dir, sim_summary_path, k8_decisions_jsonl, k8_decisions_csv


(PosixPath('/home/goncalo/KubeEnergyScheduler/kubenergysched/results_latest'),
 PosixPath('/home/goncalo/KubeEnergyScheduler/kubenergysched/results_latest/summary.csv'),
 PosixPath('/home/goncalo/KubeEnergyScheduler/decisions.jsonl'),
 PosixPath('/home/goncalo/KubeEnergyScheduler/decisions.csv'))

## 3. Load simulator traces

The simulator stores one CSV per scheduler/policy configuration with per-job
records. We consolidate them into a single `DataFrame` and derive the wait/runtime
features required for the comparisons. The aggregated metrics are loaded from the
`summary.csv` file emitted at the end of the simulation batch.


In [3]:
sim_per_job_frames: List[pd.DataFrame] = []

for csv_path in sorted(sim_dir.glob("*_results.csv")):
    if csv_path.name == "summary.csv":
        continue
    df = pd.read_csv(csv_path)
    # Ensure the common schema
    df["submit"] = pd.to_datetime(df["submit"], utc=True, errors="coerce")
    df["start"] = pd.to_datetime(df["start"], utc=True, errors="coerce")
    df["end"] = pd.to_datetime(df["end"], utc=True, errors="coerce")
    df["wait_s"] = df["wait_ms"].astype(float) / 1000.0
    df["runtime_s"] = compute_runtime_seconds(df["start"], df["end"])
    df["source"] = "simulator"

    # Add identifiers extracted from the filename (policy  configuration)
    base_name, ci_weight, batch_size, _ = csv_path.stem.rsplit("_", 3)
    df["policy"] = base_name
    df["ci_weight"] = float(ci_weight)
    df["batch_size"] = int(batch_size)
    sim_per_job_frames.append(df)

sim_per_job = pd.concat(sim_per_job_frames, ignore_index=True)
sim_per_job.head()


,sched,job_id,node,submit,start,end,wait_ms,ci_cost,wait_s,runtime_s,source,policy,ci_weight,batch_size
0,carbonscaler,job-0,gpu-0,2025-10-19 15:00:00+00:00,2025-10-19 15:00:00+00:00,2025-10-19 15:02:05+00:00,0,0.000282,0.0,125.0,simulator,carbonscaler,0.05,128
1,carbonscaler,job-1,burst-0,2025-10-19 15:00:00+00:00,2025-10-19 15:00:00+00:00,2025-10-19 15:09:46+00:00,0,0.003540,0.0,586.0,simulator,carbonscaler,0.05,128
2,carbonscaler,job-2,burst-1,2025-10-19 15:00:00+00:00,2025-10-19 15:00:00+00:00,2025-10-19 15:03:27+00:00,0,0.003450,0.0,207.0,simulator,carbonscaler,0.05,128
3,carbonscaler,job-3,gpu-0,2025-10-19 15:00:03+00:00,2025-10-19 15:02:05+00:00,2025-10-19 15:02:47+00:00,122000,0.000117,122.0,42.0,simulator,carbonscaler,0.05,128
4,carbonscaler,job-4,gpu-0,2025-10-19 15:00:04+00:00,2025-10-19 15:02:05+00:00,2025-10-19 15:11:11+00:00,121000,0.002431,121.0,546.0,simulator,carbonscaler,0.05,128


In [4]:
sim_summary = pd.read_csv(sim_summary_path)
sim_summary["source"] = "simulator"
sim_summary


,policy,ci_weight,batch_size,total_ci_cost_g,avg_ci_per_job_g,cfp_g_per_cpu_hour,avg_wait_s,makespan_s,elapsed_ms,num_jobs,alpha_mass,lookahead_min,duration_scale,duration_overrides,source
0,engine,0.05,32,2.192427,0.002192,0.004712,6302.187,20782.0,2782.0,1000,1.0,0,1.0,NaN,simulator
1,k8,0.05,32,2.152143,0.002152,0.004625,6350.138,21460.0,988.0,1000,1.0,0,1.0,NaN,simulator
2,carbonscaler,0.05,32,2.112778,0.002113,0.004541,6530.895,22121.0,2096.0,1000,1.0,0,1.0,NaN,simulator
3,themis_base,0.05,32,2.149033,0.002149,0.004619,6451.196,21554.0,1028.0,1000,1.0,0,1.0,NaN,simulator
4,het-weighted-sum,0.05,32,2.139317,0.002139,0.004598,6451.145,21739.0,1795.0,1000,1.0,0,1.0,NaN,simulator
5,engine,0.05,128,2.192427,0.002192,0.004712,6302.187,20782.0,3692.0,1000,1.0,0,1.0,NaN,simulator
6,k8,0.05,128,2.152143,0.002152,0.004625,6350.138,21460.0,1130.0,1000,1.0,0,1.0,NaN,simulator
7,carbonscaler,0.05,128,2.112778,0.002113,0.004541,6530.895,22121.0,1473.0,1000,1.0,0,1.0,NaN,simulator
8,themis_base,0.05,128,2.149033,0.002149,0.004619,6451.196,21554.0,1596.0,1000,1.0,0,1.0,NaN,simulator
9,het-weighted-sum,0.05,128,2.139317,0.002139,0.004598,6451.145,21739.0,1685.0,1000,1.0,0,1.0,NaN,simulator


## 4. Load Kubernetes replay decisions

The controller exports one JSONL record per scheduling decision. The snippet below
filters Kubernetes-native results (`result_type == "kub_result"`), converts the
timestamps, and derives wait/runtime metrics to mirror the simulator schema.


In [5]:
k8_decisions = pd.read_json(k8_decisions_jsonl, lines=True)
k8_decisions = k8_decisions.query("result_type == 'kub_result'").copy()

k8_decisions["submit"] = safe_to_datetime(k8_decisions["queued_at"])
k8_decisions["start"] = safe_to_datetime(k8_decisions["started_at"])
k8_decisions["end"] = safe_to_datetime(k8_decisions["ended_at"])

k8_decisions["wait_s"] = compute_wait_seconds(k8_decisions["submit"], k8_decisions["start"])
k8_decisions["runtime_s"] = compute_runtime_seconds(k8_decisions["start"], k8_decisions["end"])

# The replay uses a single scheduler labelled "engine" in this trace.
k8_decisions["policy"] = k8_decisions["scheduler"]
k8_decisions["source"] = "kubernetes"

k8_decisions[[
    "result_id", "scheduler", "job_id", "site", "node", "wait_s", "runtime_s", "fallback", "cost"
]].head()


/tmp/ipykernel_3244655/1776438184.py:1: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  k8_decisions = pd.read_json(k8_decisions_jsonl, lines=True)


ValueError: Expected object or value

## 5. Harmonise schemas

To compare the two environments we keep only the shared columns and annotate the
origin. The helper below produces a tidy table that concatenates simulator and
Kubernetes traces.


In [ ]:
common_cols = [
    "policy", "job_id", "site", "node", "submit", "start", "end",
    "wait_s", "runtime_s", "ci_cost", "source"
]

sim_view = sim_per_job.copy()
if "policy" not in sim_view.columns:
    sim_view["policy"] = sim_view["sched"]
if "sched" in sim_view.columns:
    sim_view = sim_view.drop(columns=["sched"])
for col in ["site", "node"]:
    if col not in sim_view:
        sim_view[col] = pd.NA
sim_view["ci_cost"] = sim_view["ci_cost"].astype(float)

k8_view = k8_decisions.copy()
if "ci_cost" not in k8_view.columns:
    k8_view["ci_cost"] = k8_view.get("cost", 0.0)

combined = pd.concat(
    [sim_view[common_cols], k8_view[common_cols]],
    ignore_index=True,
    sort=False,
)
combined.head()


## 6. Summary metrics per environment

The next cells compute aggregate indicators (number of jobs, completion ratio,
carbon impact, wait time, fallback rate) to sanity-check the two datasets before
digging into the Pareto analysis.


In [6]:
def summarise_jobs(df: pd.DataFrame) -> pd.Series:
    completed = df["end"].notna()
    submit = df["submit"].dropna()
    end_times = df.loc[completed, "end"]

    makespan = (end_times.max() - submit.min()).total_seconds() if not end_times.empty and not submit.empty else np.nan

    return pd.Series({
        "jobs": len(df),
        "completed_jobs": int(completed.sum()),
        "completion_ratio": completed.mean() if len(df) else np.nan,
        "fallback_ratio": df.get("fallback", pd.Series(dtype=float)).mean() if "fallback" in df else np.nan,
        "total_ci_cost_g": df["ci_cost"].sum(),
        "avg_ci_per_job_g": df["ci_cost"].mean(),
        "avg_wait_s": df["wait_s"].mean(),
        "avg_runtime_s": df["runtime_s"].mean(),
        "makespan_s": makespan,
    })

summary_by_source = combined.groupby(["source", "policy"]).apply(summarise_jobs).reset_index()
summary_by_source


NameError: name 'combined' is not defined

The simulator additionally exposes rich metadata (configuration weights, elapsed
simulation time) via `summary.csv`. Joining that file with the per-job aggregates
helps confirm consistency and provides the inputs for the Pareto plot.


In [ ]:
sim_summary_metrics = summary_by_source.query("source == 'simulator'").merge(
    sim_summary,
    on=["policy"],
    how="left",
    suffixes=("_jobs", "_reported"),
)
sim_summary_metrics[[
    "policy",
    "jobs",
    "total_ci_cost_g_jobs",
    "total_ci_cost_g_reported",
    "avg_wait_s_jobs",
    "avg_wait_s_reported",
]]


## 6. Evaluation metrics

We materialise the thesis metrics for carbon- and time-aware evaluation. The code below
implements the definitions from Section 6.2.2: CFP (6.2), SCI (6.3), makespan (6.4),
latency (6.5), scheduler overhead and throughput (6.6), and the average energy per job.
All quantities are normalised per batch/policy so that simulator and Kubernetes traces
can be compared directly once telemetry is populated.


In [ ]:

# Evaluate thesis metrics (CFP, SCI, makespan, latency, scheduler overhead, throughput, energy/job).
nodes_catalog = pd.read_csv(repo_root / "kubenergysched" / "config" / "nodes.csv")
sites_catalog = pd.read_csv(repo_root / "kubenergysched" / "config" / "sites.csv")

node_power_lookup = {}
node_site_lookup = {}
if {"name", "peak_power_w", "site"}.issubset(nodes_catalog.columns):
    node_power_lookup = nodes_catalog.set_index("name")["peak_power_w"].astype(float).to_dict()
    node_site_lookup = nodes_catalog.set_index("name")["site"].to_dict()

site_pue_lookup = {}
if {"id", "pue"}.issubset(sites_catalog.columns):
    site_pue_lookup = sites_catalog.set_index("id")["pue"].astype(float).to_dict()


def compute_evaluation_metrics(frame: pd.DataFrame, fallback_overhead_s: float | None = None) -> pd.Series:
    data = frame.copy()
    if data.empty:
        return pd.Series({
            "jobs_total": 0,
            "jobs_completed": 0,
            "carbon_footprint_g": np.nan,
            "carbon_footprint_kg": np.nan,
            "software_carbon_intensity_g_per_job": np.nan,
            "makespan_s": np.nan,
            "latency_s": np.nan,
            "throughput_jobs_per_hour": np.nan,
            "scheduler_overhead_s": fallback_overhead_s if fallback_overhead_s is not None else np.nan,
            "energy_per_job_wh": np.nan,
        })

    data["submit"] = safe_to_datetime(data["submit"])
    data["start"] = safe_to_datetime(data["start"])
    data["end"] = safe_to_datetime(data["end"])

    if "wait_s" in data.columns:
        data["wait_s"] = data["wait_s"].astype(float)
    else:
        data["wait_s"] = compute_wait_seconds(data["submit"], data["start"])

    if "runtime_s" in data.columns:
        data["runtime_s"] = data["runtime_s"].astype(float)
    else:
        data["runtime_s"] = compute_runtime_seconds(data["start"], data["end"])

    if "ci_cost" in data.columns:
        data["ci_cost"] = data["ci_cost"].astype(float)
    else:
        data["ci_cost"] = 0.0

    completed_mask = data["end"].notna()
    jobs_total = int(len(data))
    jobs_completed = int(completed_mask.sum())

    cfp_total_g = float(data.loc[completed_mask, "ci_cost"].sum())
    sci_g_per_job = cfp_total_g / jobs_completed if jobs_completed else np.nan

    arrival_times = data["submit"].dropna()
    completion_times = data.loc[completed_mask, "end"].dropna()
    if not arrival_times.empty and not completion_times.empty:
        makespan_s = (completion_times.max() - arrival_times.min()).total_seconds()
    else:
        makespan_s = np.nan

    latency_s = float(data["wait_s"].mean()) if len(data["wait_s"]) else np.nan

    if jobs_completed and pd.notna(makespan_s) and makespan_s > 0:
        throughput_jobs_per_hour = jobs_completed / (makespan_s / 3600.0)
    else:
        throughput_jobs_per_hour = np.nan

    energy_series = None
    for candidate in ("energy_wh", "job_energy_wh"):
        if candidate in data.columns:
            energy_series = data.loc[completed_mask, candidate].astype(float)
            break

    if energy_series is None:
        estimates = []
        for idx, row in data.loc[completed_mask].iterrows():
            node = row.get("node")
            runtime_s = row.get("runtime_s")
            power_w = node_power_lookup.get(node)
            if power_w is None or pd.isna(runtime_s) or runtime_s <= 0:
                estimates.append(np.nan)
                continue
            site_id = node_site_lookup.get(node)
            pue = site_pue_lookup.get(site_id, 1.0)
            estimates.append(power_w * pue * (runtime_s / 3600.0))
        if estimates:
            energy_series = pd.Series(estimates, index=data.loc[completed_mask].index, dtype=float)

    if energy_series is not None and energy_series.notna().any():
        total_energy_wh = float(energy_series.dropna().sum())
        energy_per_job_wh = total_energy_wh / jobs_completed if jobs_completed else np.nan
    else:
        energy_per_job_wh = np.nan

    if "scheduler_latency_s" in data.columns:
        scheduler_overhead_s = float(data["scheduler_latency_s"].astype(float).mean())
    elif "scheduler_latency_ms" in data.columns:
        scheduler_overhead_s = float(data["scheduler_latency_ms"].astype(float).mean() / 1000.0)
    elif fallback_overhead_s is not None:
        scheduler_overhead_s = float(fallback_overhead_s)
    else:
        scheduler_overhead_s = np.nan

    return pd.Series({
        "jobs_total": jobs_total,
        "jobs_completed": jobs_completed,
        "carbon_footprint_g": cfp_total_g,
        "carbon_footprint_kg": cfp_total_g / 1000.0 if pd.notna(cfp_total_g) else np.nan,
        "software_carbon_intensity_g_per_job": sci_g_per_job,
        "makespan_s": makespan_s,
        "latency_s": latency_s,
        "throughput_jobs_per_hour": throughput_jobs_per_hour,
        "scheduler_overhead_s": scheduler_overhead_s,
        "energy_per_job_wh": energy_per_job_wh,
    })


sim_overhead_lookup = {}
if not sim_summary.empty:
    temp = sim_summary.copy()
    temp["scheduler_overhead_s"] = (
        temp["elapsed_ms"].astype(float)
        / temp["num_jobs"].astype(float).clip(lower=1.0)
        / 1000.0
    )
    sim_overhead_lookup = temp.set_index(["policy", "ci_weight", "batch_size"])["scheduler_overhead_s"].to_dict()

metrics_rows = []
for (policy, ci_weight, batch_size), group in sim_per_job.groupby(["policy", "ci_weight", "batch_size"], dropna=False):
    fallback = sim_overhead_lookup.get((policy, ci_weight, batch_size))
    metrics = compute_evaluation_metrics(group, fallback_overhead_s=fallback)
    record = metrics.to_dict()
    record.update({
        "source": "simulator",
        "policy": policy,
        "ci_weight": ci_weight,
        "batch_size": batch_size,
    })
    metrics_rows.append(record)

if not k8_decisions.empty:
    if "ci_weight" not in k8_decisions.columns and "theta_c" in k8_decisions.columns:
        k8_decisions = k8_decisions.copy()
        k8_decisions["ci_weight"] = k8_decisions["theta_c"].astype(float)
    for (policy, ci_weight), group in k8_decisions.groupby(["policy", "ci_weight"], dropna=False):
        metrics = compute_evaluation_metrics(group)
        record = metrics.to_dict()
        record.update({
            "source": "kubernetes",
            "policy": policy,
            "ci_weight": ci_weight,
            "batch_size": group.get("batch_size", pd.Series([pd.NA])).iloc[0] if "batch_size" in group else pd.NA,
        })
        metrics_rows.append(record)

evaluation_metrics = pd.DataFrame(metrics_rows)

if not evaluation_metrics.empty:
    ordered_columns = [
        "source",
        "policy",
        "ci_weight",
        "batch_size",
        "jobs_total",
        "jobs_completed",
        "carbon_footprint_g",
        "carbon_footprint_kg",
        "software_carbon_intensity_g_per_job",
        "makespan_s",
        "latency_s",
        "throughput_jobs_per_hour",
        "scheduler_overhead_s",
        "energy_per_job_wh",
    ]
    evaluation_metrics = evaluation_metrics[ordered_columns].sort_values([
        "source",
        "policy",
        "ci_weight",
        "batch_size",
    ])

evaluation_metrics


### Scheduler decision mix

Aggregate the number of scheduling decisions by policy and environment to verify the replay covers the intended scenarios.


In [ ]:
decision_counts = (
    combined.groupby(["source", "policy"], dropna=False)["job_id"]
    .count()
    .reset_index(name="decisions")
)

if decision_counts.empty:
    decision_counts
else:
    pivot_counts = (
        decision_counts
        .pivot(index="policy", columns="source", values="decisions")
        .fillna(0)
        .sort_index()
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    pivot_counts.plot(kind="bar", ax=ax)
    ax.set_ylabel("Decisions")
    ax.set_title("Scheduler decisions by source")
    ax.legend(title="Source")
    fig.tight_layout()
    decisions_plot_path = figures_dir / "scheduler_decisions_by_source.png"
    fig.savefig(decisions_plot_path, dpi=300)
    pivot_counts, decisions_plot_path


## 7. Visual comparisons

We now contrast the average wait times and carbon footprint of each policy. The
Kubernetes replay (engine) is highlighted separately; because the sampled trace only
contains fallback decisions, the wait/runtime metrics stay undefined (`NaN`).


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sim_points = sim_summary_metrics.sort_values("total_ci_cost_g_reported")
ax.scatter(
    sim_points["total_ci_cost_g_reported"],
    sim_points["avg_wait_s_reported"],
    label="Simulator policies",
    c="tab:blue",
)
for _, row in sim_points.iterrows():
    ax.annotate(
        row["policy"],
        (row["total_ci_cost_g_reported"], row["avg_wait_s_reported"]),
        textcoords="offset points",
        xytext=(5, 5),
    )

k8_summary = summary_by_source.query("source == 'kubernetes'")
if not k8_summary.empty:
    ax.scatter(
        k8_summary["total_ci_cost_g"],
        k8_summary["avg_wait_s"],
        label="Kubernetes replay",
        marker="s",
        c="tab:orange",
        s=90,
    )
    for _, row in k8_summary.iterrows():
        ax.annotate(
            f"k8s:{row['policy']}",
            (row["total_ci_cost_g"], row["avg_wait_s"]),
            textcoords="offset points",
            xytext=(5, -12),
        )

ax.set_xlabel("Total CI cost (g)")
ax.set_ylabel("Average wait (s)")
ax.set_title("Carbon vs. wait trade-offs")
ax.legend()
fig.tight_layout()

tradeoff_path = figures_dir / "carbon_vs_wait_tradeoff.png"
fig.savefig(tradeoff_path, dpi=300)
tradeoff_path


## 8. Pareto-front analysis

The Pareto filter isolates the policies that are non-dominated with respect to
(total carbon, average wait). These are the options worth keeping when searching
for better baselines or new algorithms.


In [ ]:
pareto_metrics = sim_points[["total_ci_cost_g_reported", "avg_wait_s_reported"]].to_numpy()
mask = pareto_front(pareto_metrics)
pareto_candidates = sim_points.loc[mask].copy()
pareto_candidates


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    sim_points["total_ci_cost_g_reported"],
    sim_points["avg_wait_s_reported"],
    color="lightgray",
    label="All policies",
)
ax.scatter(
    pareto_candidates["total_ci_cost_g_reported"],
    pareto_candidates["avg_wait_s_reported"],
    color="tab:green",
    label="Pareto front",
)
ax.plot(
    pareto_candidates["total_ci_cost_g_reported"],
    pareto_candidates["avg_wait_s_reported"],
    color="tab:green",
    linestyle="--",
    linewidth=1,
)
for _, row in pareto_candidates.iterrows():
    ax.annotate(
        row["policy"],
        (row["total_ci_cost_g_reported"], row["avg_wait_s_reported"]),
        textcoords="offset points",
        xytext=(4, 6),
    )

ax.set_xlabel("Total CI cost (g)")
ax.set_ylabel("Average wait (s)")
ax.set_title("Simulator Pareto front")
ax.legend()
fig.tight_layout()

pareto_path = figures_dir / "simulator_pareto_front.png"
fig.savefig(pareto_path, dpi=300)
pareto_path


In [ ]:
tables_dir = figures_dir.parent / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

pareto_latex_path = tables_dir / "pareto_front.tex"
if pareto_candidates.empty:
    pareto_latex_path.write_text("% Pareto front empty\n")
else:
    pareto_candidates.to_latex(
        pareto_latex_path,
        index=False,
        float_format="%.3f",
        caption="Pareto front across simulator policies (carbon vs. wait).",
        label="tab:pareto_front",
    )
pareto_latex_path


## 9. Hooks for contextual metrics (optional)

The design document suggests extending the notebook with contextual indicators
(power traces, forecasts). When Prometheus endpoints become available, inject the
queries here and join on the `submit/start/end` timestamps stored in `combined`.


## 10. Conclusions

The consolidated tables confirm that the simulator emits complete traces for all
policies, whereas the sampled Kubernetes replay currently falls back on every job.
That discrepancy highlights the need for additional cluster-side debugging before
using the replay as a baseline. The Pareto front nonetheless identifies the
non-dominated simulator policies, informing which schedulers to evolve or replace
when collecting new baselines.
